<a href="https://colab.research.google.com/github/enlight-org-ua/ei-docs-to-markdown/blob/main/Enlight_Docs_to_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Порядок роботи:

1. Натисніть **▶** біля блоку **“Завантажити файли → отримати Markdown або ZIP”**.
2. Виберіть **PDF / Word / PowerPoint / HTML / зображення**.
3. Дочекайтесь повідомлення **✅ Процес завершено**.
4. Результат завантажиться автоматично.
5. **Після успішного завантаження результату** запустіть окремий блок **“🧹 Очистити робоче місце”**.

Підтримуються тільки: `.pdf` `.docx` `.doc` `.pptx` `.ppt` `.html` `.htm` `.png` `.jpg` `.jpeg` `.tif` `.tiff` `.bmp`


## Повторна операція

Після скачування запустіть **“🧹 Очистити робоче місце”**. Якщо потрібно конвертувати нові файли, повторіть процес з самого початку.


In [11]:
#@title ▶ Завантажити файли → отримати Markdown або ZIP
#@markdown 1. Натисніть ▶ зліва від цього блоку.
#@markdown 2. Виберіть PDF, Word, PowerPoint, HTML або зображення. ZIP-архіви не приймаються.
#@markdown 3. Дочекайтесь статусу “✅ Процес завершено” і завантажте файл результату.
#@markdown 4. Після успішного завантаження запустіть окремий блок “🧹 Очистити робоче місце”.

# ============================================================
# Авторство та права на використання
# Автор інструменту: ХОМГО «Освічена ініціатива» / NGO “Enlightening Initiative”
# Website: https://enlight.org.ua
# Telegram: https://t.me/enlight_hub
# Instagram: https://www.instagram.com/ngo.ei/
# Facebook: https://www.facebook.com/enlighteninginitiative/
# LinkedIn: https://ua.linkedin.com/company/ngo-enlightening-initiative
#
# Дозволено використовувати, копіювати, адаптувати та розповсюджувати
# цей notebook для освітніх, гуманітарних, некомерційних та
# організаційних цілей за умови збереження зазначення авторства.
#
# Не дозволено без окремої письмової згоди автора:
# - видавати цей notebook або його суттєві частини за власну розробку;
# - продавати notebook як окремий комерційний продукт;
# - прибирати згадку про ХОМГО «Освічена ініціатива» як автора.
# ============================================================

import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

import re
import sys
import csv
import time
import shutil
import zipfile
import logging
import warnings
import threading
import subprocess
from pathlib import Path
from datetime import datetime
from contextlib import redirect_stdout, redirect_stderr
from IPython.display import display, HTML, clear_output

PAGES_PER_CHUNK = 10
SPLIT_PDFS_OVER_PAGES = 10

SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".docx",
    ".doc",
    ".pptx",
    ".ppt",
    ".html",
    ".htm",
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp",
}

warnings.filterwarnings("ignore")

for name in [
    "huggingface_hub",
    "huggingface_hub.utils._auth",
    "huggingface_hub.utils._http",
    "docling",
    "docling.pipeline",
    "docling.pipeline.standard_pdf_pipeline",
    "docling.models",
    "docling_core",
    "pypdf",
]:
    logging.getLogger(name).setLevel(logging.CRITICAL)

logging.getLogger().setLevel(logging.ERROR)

WORKSPACE_DIR = Path("/content/EI_DOCLING_CONVERTER")
UPLOAD_DIR = WORKSPACE_DIR / "01_UPLOAD_FILES_HERE"
CHUNKS_DIR = WORKSPACE_DIR / "02_TEMP_PDF_CHUNKS"
RESULTS_DIR = WORKSPACE_DIR / "03_MARKDOWN_RESULTS"
DOWNLOAD_DIR = WORKSPACE_DIR / "04_DOWNLOAD_READY"
LOGS_DIR = WORKSPACE_DIR / "05_LOGS"
PDF_CHUNKS_DIR = CHUNKS_DIR / "PDF_CHUNKS"

ALL_FIXED_DIRS = [
    UPLOAD_DIR,
    CHUNKS_DIR,
    RESULTS_DIR,
    DOWNLOAD_DIR,
    LOGS_DIR,
]


def show_status(title, message="", kind="info"):
    styles = {
        "info": ["#333843", "#F3EEE8", "ℹ️"],
        "work": ["#2563EB", "#EFF6FF", "⏳"],
        "ok": ["#067647", "#ECFDF3", "✅"],
        "warn": ["#B54708", "#FFFAEB", "⚠️"],
        "error": ["#B42318", "#FEF3F2", "❌"],
    }

    border, bg, icon = styles.get(kind, styles["info"])

    display(HTML(f"""
    <div style="
        border:1px solid #E5E7EB;
        border-left:7px solid {border};
        background:{bg};
        padding:14px 16px;
        margin:12px 0;
        border-radius:12px;
        font-family:Arial,sans-serif;
        color:#111827;">
      <div style="font-size:17px;font-weight:700;margin-bottom:4px;">
        {icon} {title}
      </div>
      <div style="font-size:14px;line-height:1.45;">
        {message}
      </div>
    </div>
    """))


def show_folder_hint():
    display(HTML("""
    <div style="
        border:1px dashed #9CA3AF;
        background:#FFFFFF;
        padding:12px 14px;
        margin:8px 0 14px 0;
        border-radius:10px;
        font-family:Arial,sans-serif;
        font-size:14px;
        color:#111827;">
      <b>Де файли?</b><br>
      Зліва в Colab відкрийте <b>Files / Файли</b> → <code>EI_DOCLING_CONVERTER</code>.<br><br>
      <code>01_UPLOAD_FILES_HERE</code> — прийняті завантажені файли<br>
      <code>02_TEMP_PDF_CHUNKS</code> — тимчасові PDF-шматки<br>
      <code>03_MARKDOWN_RESULTS</code> — Markdown-результати<br>
      <code>04_DOWNLOAD_READY</code> — файл для скачування<br>
      <code>05_LOGS</code> — conversion_log.csv
    </div>
    """))


def heartbeat(label, stop_event, every_seconds=15):
    started = time.time()

    while not stop_event.wait(every_seconds):
        elapsed = int(time.time() - started)
        print(f"   ⏳ Процес працює: {label}. Минуло приблизно {elapsed} сек. Не закривайте вкладку.")


def reset_workspace():
    if WORKSPACE_DIR.exists():
        shutil.rmtree(WORKSPACE_DIR, ignore_errors=True)

    for folder in ALL_FIXED_DIRS:
        folder.mkdir(parents=True, exist_ok=True)

    PDF_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)


def safe_filename(name):
    name = Path(name).name.strip()
    name = re.sub(r'[\\/:"*?<>|]+', "_", name)
    name = re.sub(r"\s+", " ", name)
    return name or "document"


def unique_path(path):
    if not path.exists():
        return path

    parent = path.parent
    stem = path.stem
    suffix = path.suffix
    counter = 2

    while True:
        candidate = parent / f"{stem}_{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


clear_output(wait=True)

show_status(
    "Починаємо",
    "Готуємо фіксовану структуру папок. Усі файли будуть тільки в /content/EI_DOCLING_CONVERTER.",
    "work",
)

reset_workspace()
show_folder_hint()

try:
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from pypdf import PdfReader, PdfWriter
except Exception:
    show_status(
        "Встановлюємо компоненти",
        "Автоматично встановлюємо Docling і pypdf. Зачекайте, будь ласка. Екран може не змінюватися кілька хвилин.",
        "work",
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "docling",
        "pypdf",
    ])

    warnings.filterwarnings("ignore")

    for name in [
        "huggingface_hub",
        "huggingface_hub.utils._auth",
        "huggingface_hub.utils._http",
        "docling",
        "docling.pipeline",
        "docling.pipeline.standard_pdf_pipeline",
        "docling.models",
        "docling_core",
        "pypdf",
    ]:
        logging.getLogger(name).setLevel(logging.CRITICAL)

    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from pypdf import PdfReader, PdfWriter


def create_stable_converter():
    try:
        pdf_options = PdfPipelineOptions()
        pdf_options.do_table_structure = False
        pdf_options.do_ocr = False

        return DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
            }
        )
    except Exception:
        return DocumentConverter()


def get_pdf_page_count(pdf_path):
    return len(PdfReader(str(pdf_path)).pages)


def split_pdf_into_chunks(pdf_path, chunks_root, pages_per_chunk=10):
    reader = PdfReader(str(pdf_path))
    page_count = len(reader.pages)

    safe_stem = safe_filename(pdf_path.stem)
    file_chunks_dir = chunks_root / safe_stem
    file_chunks_dir.mkdir(parents=True, exist_ok=True)

    chunks = []

    for start_index in range(0, page_count, pages_per_chunk):
        end_index = min(start_index + pages_per_chunk, page_count)

        writer = PdfWriter()

        for page_index in range(start_index, end_index):
            writer.add_page(reader.pages[page_index])

        start_page = start_index + 1
        end_page = end_index

        chunk_path = unique_path(
            file_chunks_dir / f"{safe_stem}__pages_{start_page:04d}-{end_page:04d}.pdf"
        )

        with open(chunk_path, "wb") as f:
            writer.write(f)

        chunks.append((chunk_path, start_page, end_page))

    return page_count, chunks


def convert_path_to_markdown(converter, source_path):
    with open(os.devnull, "w") as devnull:
        with redirect_stdout(devnull), redirect_stderr(devnull):
            result = converter.convert(str(source_path))
            markdown = result.document.export_to_markdown()

    return markdown


clear_output(wait=True)

show_status(
    "Готово до завантаження файлів",
    "Виберіть PDF, Word, PowerPoint, HTML або зображення. ZIP-архіви не приймаються.",
    "ok",
)

show_folder_hint()

from google.colab import files

show_status(
    "Виберіть файли у новому вікні",
    "Файли будуть завантажені одразу в папку <code>01_UPLOAD_FILES_HERE</code>. "
    "Непідтримувані формати будуть одразу видалені з робочого місця. "
    "ZIP-архіви як вхідні файли не підтримуються.",
    "info",
)

# Важливо:
# files.upload() у Colab зберігає файли в поточну робочу папку.
# Тому перед upload переходимо саме в UPLOAD_DIR.
previous_cwd = os.getcwd()
os.chdir(str(UPLOAD_DIR))

try:
    uploaded = files.upload()
finally:
    os.chdir(previous_cwd)

if not uploaded:
    show_status(
        "Файли не завантажено",
        "Запустіть блок ще раз і виберіть файли.",
        "error",
    )
    raise SystemExit

accepted_uploads = []
rejected_uploads = []

for filename in uploaded.keys():
    original_path = UPLOAD_DIR / filename
    safe_name = safe_filename(filename)
    extension = Path(safe_name).suffix.lower()

    if extension not in SUPPORTED_EXTENSIONS:
        if original_path.exists():
            original_path.unlink()

        rejected_uploads.append({
            "filename": safe_name,
            "reason": "Формат не підтримується і не був завантажений у робоче місце",
        })
        continue

    target_path = unique_path(UPLOAD_DIR / safe_name)

    if original_path.exists():
        if original_path.resolve() != target_path.resolve():
            original_path.rename(target_path)
    else:
        rejected_uploads.append({
            "filename": safe_name,
            "reason": "Файл не знайдено після завантаження",
        })
        continue

    accepted_uploads.append(target_path.name)

clear_output(wait=True)

if rejected_uploads:
    rejected_html = "<br>".join([
        f"<code>{item['filename']}</code> — {item['reason']}"
        for item in rejected_uploads
    ])

    show_status(
        "Частину файлів відхилено",
        "Ці файли не підтримуються і не були збережені в робочому місці:<br>" + rejected_html,
        "warn",
    )

if not accepted_uploads:
    show_status(
        "Немає підтримуваних файлів",
        "Жоден із вибраних файлів не має підтримуваного формату. "
        "Завантажте PDF, Word, PowerPoint, HTML або зображення.",
        "error",
    )
    raise SystemExit

show_status(
    "Файли перевірено",
    f"Прийнято підтримуваних файлів: <b>{len(accepted_uploads)}</b>.<br>"
    f"Відхилено непідтримуваних файлів: <b>{len(rejected_uploads)}</b>.<br>"
    "Прийняті файли збережено в папці <code>01_UPLOAD_FILES_HERE</code>.",
    "ok",
)

show_folder_hint()

input_files = []

for path in UPLOAD_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
        input_files.append(path)

input_files = sorted(set(input_files))
input_count = len(input_files)

if not input_files:
    show_status(
        "Немає файлів для конвертації",
        "Не знайдено підтримуваних документів. Завантажте PDF, Word, PowerPoint, HTML або зображення.",
        "error",
    )
    raise SystemExit

clear_output(wait=True)

if input_count == 1:
    output_rule = "Знайдено один документ — після конвертації завантажиться один `.md` файл без архіву."
else:
    output_rule = "Знайдено більше одного документа — після конвертації завантажиться ZIP-архів із `.md` файлами та `conversion_log.csv`."

show_status(
    "Починаємо конвертацію",
    f"Знайдено документів для обробки: <b>{input_count}</b>.<br>"
    f"{output_rule}<br>"
    f"PDF понад <b>{SPLIT_PDFS_OVER_PAGES}</b> сторінок буде розбито на частини по <b>{PAGES_PER_CHUNK}</b> сторінок.",
    "work",
)

show_folder_hint()

converter = create_stable_converter()
conversion_log = []
success_count = 0
error_count = 0
partial_count = 0
created_outputs = []

for index, source_path in enumerate(input_files, start=1):
    started_at = datetime.now().isoformat(timespec="seconds")

    try:
        relative_path = source_path.relative_to(UPLOAD_DIR)
    except Exception:
        relative_path = Path(source_path.name)

    output_subdir = RESULTS_DIR / relative_path.parent
    output_subdir.mkdir(parents=True, exist_ok=True)

    output_md_path = unique_path(output_subdir / f"{source_path.stem}.md")

    print(f"\n[{index}/{input_count}] ⏳ Обробляється: {relative_path}")
    print("   Процес працює. Не закривайте вкладку.")

    status = "error"
    error = ""
    page_count = ""
    chunks_total = 0
    chunks_failed = 0
    processing_mode = "direct"

    try:
        if source_path.suffix.lower() == ".pdf":
            try:
                page_count = get_pdf_page_count(source_path)
            except Exception as page_error:
                page_count = ""
                print(f"   ⚠️ Не вдалося порахувати сторінки PDF. Конвертуємо як один файл. Деталь: {page_error}")

            if isinstance(page_count, int) and page_count > SPLIT_PDFS_OVER_PAGES:
                processing_mode = "chunked_pdf"

                print(f"   📄 PDF має {page_count} сторінок.")
                print(f"   ✂️ Розбиваємо на частини по {PAGES_PER_CHUNK} сторінок...")

                page_count, chunks = split_pdf_into_chunks(
                    source_path,
                    PDF_CHUNKS_DIR,
                    PAGES_PER_CHUNK,
                )

                chunks_total = len(chunks)

                print(f"   ✅ Розбито на {chunks_total} частин.")
                print("   Починаємо конвертацію частин:")

                merged_parts = []

                for chunk_index, (chunk_path, start_page, end_page) in enumerate(chunks, start=1):
                    label = f"{relative_path} — частина {chunk_index}/{chunks_total}, сторінки {start_page}-{end_page}"

                    print(f"      [{chunk_index}/{chunks_total}] ⏳ Конвертуємо сторінки {start_page}-{end_page}...")

                    stop_event = threading.Event()
                    thread = threading.Thread(
                        target=heartbeat,
                        args=(label, stop_event),
                        daemon=True,
                    )
                    thread.start()

                    try:
                        chunk_markdown = convert_path_to_markdown(converter, chunk_path)

                        if not chunk_markdown.strip():
                            raise RuntimeError("Docling не повернув текст для цієї частини.")

                        merged_parts.append(
                            f"<!-- Сторінки {start_page}-{end_page} -->\n\n"
                            f"{chunk_markdown.strip()}"
                        )

                        print(f"      ✅ Готово: сторінки {start_page}-{end_page}")

                    except Exception:
                        chunks_failed += 1

                        print(f"      ❌ Помилка в частині {chunk_index}/{chunks_total}, сторінки {start_page}-{end_page}")

                        merged_parts.append(
                            f"<!-- Сторінки {start_page}-{end_page}: ПОМИЛКА КОНВЕРТАЦІЇ -->\n\n"
                            f"> Не вдалося конвертувати цю частину. Деталі дивіться у conversion_log.csv."
                        )

                    finally:
                        stop_event.set()

                if not merged_parts or chunks_failed == chunks_total:
                    raise RuntimeError("Не вдалося конвертувати жодну частину PDF.")

                print("   🔗 Склеюємо частини в один Markdown-файл...")

                final_markdown = (
                    f"<!-- Конвертовано з PDF: {relative_path} -->\n"
                    f"<!-- Загальна кількість сторінок: {page_count} -->\n"
                    f"<!-- PDF було розбито на {chunks_total} частин по {PAGES_PER_CHUNK} сторінок -->\n\n"
                    + "\n\n---\n\n".join(merged_parts)
                    + "\n"
                )

                with open(output_md_path, "w", encoding="utf-8") as f:
                    f.write(final_markdown)

                if chunks_failed:
                    status = "partial"
                    partial_count += 1
                    error = f"PDF склеєно, але не вдалося конвертувати частин: {chunks_failed} з {chunks_total}"
                    print(f"   ⚠️ Склеєно Markdown, але частин з помилками: {chunks_failed}/{chunks_total}")
                else:
                    status = "ok"
                    error = ""
                    print("   ✅ Усі частини конвертовано.")

                success_count += 1
                created_outputs.append(output_md_path)

                print(f"   ✅ Готово: {output_md_path.relative_to(RESULTS_DIR)}")

            else:
                processing_mode = "direct_pdf"

                stop_event = threading.Event()
                thread = threading.Thread(
                    target=heartbeat,
                    args=(str(relative_path), stop_event),
                    daemon=True,
                )
                thread.start()

                try:
                    markdown = convert_path_to_markdown(converter, source_path)

                    if not markdown.strip():
                        raise RuntimeError("Docling не повернув текст. Можливо, PDF є сканованим зображенням.")

                    with open(output_md_path, "w", encoding="utf-8") as f:
                        f.write(markdown)

                    status = "ok"
                    error = ""
                    success_count += 1
                    created_outputs.append(output_md_path)

                    print(f"   ✅ Готово: {output_md_path.relative_to(RESULTS_DIR)}")

                finally:
                    stop_event.set()

        else:
            processing_mode = "direct_non_pdf"

            stop_event = threading.Event()
            thread = threading.Thread(
                target=heartbeat,
                args=(str(relative_path), stop_event),
                daemon=True,
            )
            thread.start()

            try:
                markdown = convert_path_to_markdown(converter, source_path)

                if not markdown.strip():
                    raise RuntimeError("Docling не повернув текст.")

                with open(output_md_path, "w", encoding="utf-8") as f:
                    f.write(markdown)

                status = "ok"
                error = ""
                success_count += 1
                created_outputs.append(output_md_path)

                print(f"   ✅ Готово: {output_md_path.relative_to(RESULTS_DIR)}")

            finally:
                stop_event.set()

    except Exception as e:
        status = "error"
        error = str(e)
        error_count += 1

        print(f"   ❌ Не вдалося обробити: {relative_path}. Деталі будуть у conversion_log.csv")

    conversion_log.append({
        "source_file": str(relative_path),
        "output_file": str(output_md_path.relative_to(RESULTS_DIR)) if output_md_path.exists() else "",
        "status": status,
        "processing_mode": processing_mode,
        "page_count": page_count,
        "chunks_total": chunks_total,
        "chunks_failed": chunks_failed,
        "error": error,
        "started_at": started_at,
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    })

log_csv_path = LOGS_DIR / "conversion_log.csv"

with open(log_csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "source_file",
            "output_file",
            "status",
            "processing_mode",
            "page_count",
            "chunks_total",
            "chunks_failed",
            "error",
            "started_at",
            "finished_at",
        ],
    )
    writer.writeheader()
    writer.writerows(conversion_log)

download_path = None
download_kind = None

if input_count == 1 and len(created_outputs) == 1:
    single_md = created_outputs[0]
    fixed_download_md = DOWNLOAD_DIR / single_md.name

    shutil.copy2(single_md, fixed_download_md)

    download_path = fixed_download_md
    download_kind = "md"

elif input_count == 1 and len(created_outputs) == 0:
    fixed_log = DOWNLOAD_DIR / "conversion_log.csv"

    shutil.copy2(log_csv_path, fixed_log)

    download_path = fixed_log
    download_kind = "log"

else:
    show_status(
        "Пакуємо результати",
        "Створюємо ZIP-архів у папці 04_DOWNLOAD_READY. У ZIP буде лише Markdown і conversion_log.csv.",
        "work",
    )

    ZIP_STAGING_DIR = DOWNLOAD_DIR / "ZIP_CONTENT"

    if ZIP_STAGING_DIR.exists():
        shutil.rmtree(ZIP_STAGING_DIR, ignore_errors=True)

    ZIP_STAGING_DIR.mkdir(parents=True, exist_ok=True)

    for md_path in sorted(created_outputs):
        relative_md = md_path.relative_to(RESULTS_DIR)
        target_md = ZIP_STAGING_DIR / relative_md

        target_md.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(md_path, target_md)

    shutil.copy2(log_csv_path, ZIP_STAGING_DIR / "conversion_log.csv")

    zip_path = Path(
        shutil.make_archive(
            str(DOWNLOAD_DIR / "markdown_results"),
            "zip",
            ZIP_STAGING_DIR,
        )
    )

    shutil.rmtree(ZIP_STAGING_DIR, ignore_errors=True)

    download_path = zip_path
    download_kind = "zip"

clear_output(wait=True)

if download_kind == "md":
    show_status(
        "Процес завершено",
        "Успішно конвертовано один документ. Якщо це був великий PDF, усі частини вже склеєні в один Markdown-файл. "
        "Файл лежить у <code>04_DOWNLOAD_READY</code>. Зараз браузер завантажить Markdown-файл без архіву.",
        "ok",
    )

elif download_kind == "zip":
    show_status(
        "Процес завершено",
        f"Успішно конвертовано: <b>{success_count}</b>. "
        f"Частково конвертовано: <b>{partial_count}</b>. "
        f"Помилки: <b>{error_count}</b>. "
        "Якщо серед файлів були великі PDF, кожен із них уже склеєний у власний Markdown-файл. "
        "Файл лежить у <code>04_DOWNLOAD_READY</code>. Зараз браузер завантажить ZIP-архів.",
        "ok" if success_count > 0 else "warn",
    )

elif download_kind == "log":
    show_status(
        "Процес завершено з помилкою",
        "Документ не вдалося конвертувати. У <code>04_DOWNLOAD_READY</code> буде завантажено conversion_log.csv з деталями помилки.",
        "warn",
    )

show_folder_hint()

print("Робочий каталог:", WORKSPACE_DIR)
print("Файл для скачування:", download_path)
print("Тип результату:", download_kind)

if download_kind == "zip":
    with zipfile.ZipFile(download_path, "r") as zf:
        zip_contents = zf.namelist()

    print("\nФайли в ZIP:")
    for item in zip_contents:
        print(" -", item)

print("\nЯкщо автоматичне скачування не почалося, зліва відкрийте Files / Файли → EI_DOCLING_CONVERTER → 04_DOWNLOAD_READY.")
print("Після успішного скачування запустіть окремий блок нижче: 🧹 Очистити робоче місце.")

files.download(str(download_path))

show_status(
    "Файл передано на скачування",
    "Перевірте, що файл успішно завантажився на комп’ютер. "
    "Потім запустіть окремий блок <b>🧹 Очистити робоче місце</b>, щоб видалити вміст усіх робочих папок у Colab.<br><br>"
    "Якщо потрібно повторити операцію, після очищення знову запустіть основний блок зверху і виконайте всі кроки спочатку.",
    "ok",
)

Робочий каталог: /content/EI_DOCLING_CONVERTER
Файл для скачування: /content/EI_DOCLING_CONVERTER/04_DOWNLOAD_READY/markdown_results.zip
Тип результату: zip

Файли в ZIP:
 - chatbot_instructions_enlight-2_2.md
 - conversion_log.csv
 - referral_ternopil_2.md
 - referral_kharkiv_2.md

Якщо автоматичне скачування не почалося, зліва відкрийте Files / Файли → EI_DOCLING_CONVERTER → 04_DOWNLOAD_READY.
Після успішного скачування запустіть окремий блок нижче: 🧹 Очистити робоче місце.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
#@title 🧹 Очистити робоче місце
#@markdown Запускайте цей блок **тільки після того, як результат успішно завантажився на ваш комп’ютер**.
#@markdown Він очистить вміст усіх п’яти папок, але залишить самі папки з фіксованими назвами.


# ============================================================
# Авторство та права на використання
# Автор інструменту: ХОМГО «Освічена ініціатива» / NGO “Enlightening Initiative”
# Website: https://enlight.org.ua
# Telegram: https://t.me/enlight_hub
# Instagram: https://www.instagram.com/ngo.ei/
# Facebook: https://www.facebook.com/enlighteninginitiative/
# LinkedIn: https://ua.linkedin.com/company/ngo-enlightening-initiative
#
# Дозволено використовувати, копіювати, адаптувати та розповсюджувати
# цей notebook для освітніх, гуманітарних, некомерційних та
# організаційних цілей за умови збереження зазначення авторства.
#
# Не дозволено без окремої письмової згоди автора:
# - видавати цей notebook або його суттєві частини за власну розробку;
# - продавати notebook як окремий комерційний продукт;
# - прибирати згадку про ХОМГО «Освічена ініціатива» як автора.
# ============================================================

import shutil
from pathlib import Path
from IPython.display import display, HTML

WORKSPACE_DIR = Path("/content/EI_DOCLING_CONVERTER")
FOLDERS_TO_CLEAR = [
    WORKSPACE_DIR / "01_UPLOAD_FILES_HERE",
    WORKSPACE_DIR / "02_TEMP_PDF_CHUNKS",
    WORKSPACE_DIR / "03_MARKDOWN_RESULTS",
    WORKSPACE_DIR / "04_DOWNLOAD_READY",
    WORKSPACE_DIR / "05_LOGS",
]

def show_cleanup_status(title, message="", kind="ok"):
    styles={"ok":["#067647","#ECFDF3","✅"],"warn":["#B54708","#FFFAEB","⚠️"]}
    border,bg,icon=styles.get(kind,styles["ok"])
    display(HTML(f"""
    <div style="border:1px solid #E5E7EB;border-left:7px solid {border};background:{bg};padding:14px 16px;margin:12px 0;border-radius:12px;font-family:Arial,sans-serif;color:#111827;">
      <div style="font-size:17px;font-weight:700;margin-bottom:4px;">{icon} {title}</div>
      <div style="font-size:14px;line-height:1.45;">{message}</div>
    </div>"""))

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
cleared=[]
for folder in FOLDERS_TO_CLEAR:
    if folder.exists():
        shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True, exist_ok=True)
    cleared.append(str(folder.relative_to(WORKSPACE_DIR)))

show_cleanup_status(
    "Робоче місце очищено",
    "Вміст усіх п’яти папок видалено. Фіксована структура папок залишилась порожньою:<br>"
    + "<br>".join([f"<code>{name}</code>" for name in cleared])
    + "<br><br>Файл, який уже був завантажений на ваш комп’ютер, не видаляється."
    + "<br><br><b>Щоб повторити операцію:</b> знову запустіть основний блок <b>▶ Завантажити файли → отримати Markdown або ZIP</b> і виконайте процес спочатку.",
    "ok"
)